In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cp /content/drive/MyDrive/fer2013plus.zip /content/

In [ ]:
!mkdir fer2013plus/

In [ ]:
!unzip fer2013plus.zip -d /content/fer2013plus/

Streaming output truncated to the last 5000 lines.
  inflating: /content/fer2013plus/fer2013plus/fer2013/train/sadness/fer0017527.png  
  inflating: /content/fer2013plus/fer2013plus/fer2013/train/sadness/fer0017536.png  
  inflating: /content/fer2013plus/fer2013plus/fer2013/train/sadness/fer0017542.png  
  inflating: /content/fer2013plus/fer2013plus/fer2013/train/sadness/fer0017574.png  
  inflating: /content/fer2013plus/fer2013plus/fer2013/train/sadness/fer0017577.png  
  inflating: /content/fer2013plus/fer2013plus/fer2013/train/sadness/fer0017579.png  
  inflating: /content/fer2013plus/fer2013plus/fer2013/train/sadness/fer0017584.png  
  inflating: /content/fer2013plus/fer2013plus/fer2013/train/sadness/fer0017610.png  
  inflating: /content/fer2013plus/fer2013plus/fer2013/train/sadness/fer0017616.png  
  inflating: /content/fer2013plus/fer2013plus/fer2013/train/sadness/fer0017623.png  
  inflating: /content/fer2013plus/fer2013plus/fer2013/train/sadness/fer0017633.png  
  inflating: /

## Defining architechure and train function


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

transform_train = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transform_test = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.ImageFolder('/content/fer2013plus/fer2013plus/fer2013/train/', transform=transform_train)
test_dataset  = datasets.ImageFolder('/content/fer2013plus/fer2013plus/fer2013/test/', transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

class FER_DCNN(nn.Module):
    def __init__(self):
        super(FER_DCNN, self).__init__()

        def conv_block(in_f, out_f, dropout_rate):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ELU(),
                nn.Conv2d(out_f, out_f, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_f),
                nn.ELU(),
                nn.MaxPool2d(2),
                nn.Dropout(dropout_rate)
            )

        # 1. Conv block count: 3 sets
        # 2. Filter counts: 64, 128, 256
        # 3. Dropout (conv): Rate 0.3 for all blocks
        self.features = nn.Sequential(
            conv_block(1, 64, 0.3),   # Block 1
            conv_block(64, 128, 0.3),  # Block 2
            conv_block(128, 256, 0.3)  # Block 3
        )

        # 4. Dense layers: 1 dense, 128 neurons
        # Spatial size logic: 48 -> 24 -> 12 -> 6 (after 3 MaxPools)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 128),
            nn.BatchNorm1d(128),
            nn.ELU(),
            nn.Dropout(0.4),
            nn.Linear(128, 8)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FER_DCNN().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  #implemented

optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

def train_epoch(model, loader):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels) # Calculate loss
            _, preds = torch.max(outputs, 1)

            total_loss += loss.item() # Accumulate loss
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total, total_loss / len(loader) # Return both accuracy and loss

## Training Model

In [ ]:
# Training loop
best_acc, best_val_loss = 0, float('inf')
patience, no_improve_count, epoch = 10, 0, 0

while True:
    train_loss= train_epoch(model, train_loader)
    val_acc, val_loss = evaluate(model, test_loader)

    scheduler.step(val_loss)

    if val_acc > best_acc + 1e-4:
        best_acc = val_acc
        no_improve_count = 0
        torch.save(model.state_dict(), "best_fer_model.pth")
    else:
        no_improve_count += 1

    print(f"Epoch {epoch+1}: TrainLoss={train_loss:.4f}  ValLoss={val_loss:.4f}  "
          f"ValAcc={val_acc:.4f}  NoImprove={no_improve_count}/{patience}")

    if no_improve_count >= patience:
        print(f"Converged at epoch {epoch+1}. Best Val Acc: {best_acc:.4f}")
        break

    epoch += 1

Epoch 1: TrainLoss=1.4188  ValLoss=1.2706  ValAcc=0.6326  NoImprove=0/10
Epoch 2: TrainLoss=1.1953  ValLoss=1.1254  ValAcc=0.6997  NoImprove=0/10
Epoch 3: TrainLoss=1.1318  ValLoss=1.1037  ValAcc=0.7074  NoImprove=0/10
Epoch 4: TrainLoss=1.0913  ValLoss=1.0601  ValAcc=0.7302  NoImprove=0/10
Epoch 5: TrainLoss=1.0642  ValLoss=1.0293  ValAcc=0.7439  NoImprove=0/10
Epoch 6: TrainLoss=1.0391  ValLoss=1.0138  ValAcc=0.7566  NoImprove=0/10
Epoch 7: TrainLoss=1.0217  ValLoss=1.0103  ValAcc=0.7580  NoImprove=0/10
Epoch 8: TrainLoss=1.0013  ValLoss=0.9911  ValAcc=0.7641  NoImprove=0/10
Epoch 9: TrainLoss=0.9858  ValLoss=0.9848  ValAcc=0.7652  NoImprove=0/10
Epoch 10: TrainLoss=0.9682  ValLoss=0.9952  ValAcc=0.7594  NoImprove=1/10
Epoch 11: TrainLoss=0.9547  ValLoss=0.9988  ValAcc=0.7583  NoImprove=2/10
Epoch 12: TrainLoss=0.9399  ValLoss=0.9620  ValAcc=0.7831  NoImprove=0/10
Epoch 13: TrainLoss=0.9304  ValLoss=0.9483  ValAcc=0.7855  NoImprove=0/10
Epoch 14: TrainLoss=0.9155  ValLoss=0.9402  Val